# Prerequisites

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

'wget' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [2]:
import sys
import os

# None otherwise
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable :", sys.executable)
print("Python version    :", sys.version)
print("PYSPARK_PYTHON    :", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER    :", os.environ.get("PYSPARK_DRIVER_PYTHON"))
print("JAVA_HOME         :", os.environ.get("JAVA_HOME"))

Python executable : c:\Users\jadep\OneDrive\Documents\PlatformIO\big_data\ing5\big_data\Scripts\python.exe
Python version    : 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
PYSPARK_PYTHON    : c:\Users\jadep\OneDrive\Documents\PlatformIO\big_data\ing5\big_data\Scripts\python.exe
PYSPARK_DRIVER    : c:\Users\jadep\OneDrive\Documents\PlatformIO\big_data\ing5\big_data\Scripts\python.exe
JAVA_HOME         : C:\Program Files\Eclipse Adoptium\jdk-17.0.14.7-hotspot\


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .master("local[*]") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Defind the rdd
rdd = sc.textFile("./around_the_world_in_80_days.txt") #("./02_pyspark_jupyter_docker/around_the_world_in_80_days.txt")

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

The function `flatMap` return another rdd, with the function given applied.

So it returned the type of `words` : PythonRDD

And it was created from another PythonRDD, line 59, and the code was in scala.

<ADD EXPLANATION HERE>

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the above command
words.collect()

`words` is like `rdd`, but every single word is separated, and every single string (sentence) is now a list of words (that make up the sentence).

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.

In [ ]:
rdd.flatMap?

The function `flatMap` return another rdd, with the function given applied. Then it 'flatten' is result, so it changes the dimension from X to 1.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [ ]:
# a. count the occurence of each word
words_occurence = words.reduceByKey(lambda  w1, w2: w1 + w2)

for word, count in words_occurence.collect():
    print(f"'{word}' is said {count} times")

In [ ]:
# b. a common first step in text analysis, change all capital letters to lower case
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word.lower(), 1))

words_occurence = words.reduceByKey(lambda  w1, w2: w1 + w2)

for word, count in words_occurence.collect():
    print(f"'{word}' is said {count} times")

In [ ]:
# c. eliminate the stop words.
stop_words = [
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 
    'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 
    'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 
    'can', 'cannot', 'could', 'couldn', "couldn't", 'did', 'didn', "didn't", 
    'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 
    'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', 
    "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 
    'he', "he'd", "he'll", "he's", 'her', 'here', "here's", 'hers', 
    'herself', 'him', 'himself', 'his', 'how', "how's", 'i', "i'd", "i'll", 
    "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it's", 
    'its', 'itself', "let's", 'me', 'more', 'most', 'mustn', "mustn't", 'my', 
    'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 
    'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 
    'same', 'shan', "shan't", 'she', "she'd", "she'll", "she's", 'should', 
    'shouldn', "shouldn't", 'so', 'some', 'such', 'than', 'that', "that's", 
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 
    "there's", 'these', 'they', "they'd", "they'll", "they're", "they've", 
    'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 
    'was', 'wasn', "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', 
    'weren', "weren't", 'what', "what's", 'when', "when's", 'where',
    "where's", 'which', 'while', 'who', "who's", 'whom', 'why', "why's", 
    'with', 'won', "won't", 'would', 'wouldn', "wouldn't", 'you', "you'd", 
    "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves'
]

words_without_stop_words = words_occurence.filter(lambda word: word[0] not in stop_words)

for word, count in words_without_stop_words.collect():
    print(f"'{word}' is said {count} times")

In [ ]:
# d. sort in alphabetical order
sorted_words = words_without_stop_words.sortByKey()

for word, count in sorted_words.collect():
    print(f"'{word}' is said {count} times")

In [ ]:
# e. sort descending by word frequency
sorted_words = words_without_stop_words.sortBy(lambda word: word[1], ascending=False)

for word, count in sorted_words.collect():
    print(f"'{word}' is said {count} times")

In [ ]:
# f. remove punctuations and blank spaces
punctuations = [
    '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', 
    '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', 
    '`', '{', '|', '}', '~', '«', '»', '“', '”', '‘', '’', '–', '—', '…',
    ' ', '  ', '   ', '§', ''
]

words_cleaned = sorted_words.filter(lambda word: word[0] not in punctuations)

# must remove ponctuation on word alone, like "“i"

for word, count in words_cleaned.collect():
    print(f"'{word}' is said {count} times")

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30), ("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # for each element, you keep the same key (names like Brooke), but change the number into a tuple (20 become (20, 1)) 
  .map(lambda x: (x[0], (x[1], 1)))
  # regroup elements that have the same key (name) by adding them (so it also count them)
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # for each element, you divided the sum of age by the number of age, so it calculate the average of age
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

# 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible

In [3]:
# Set
stop_words = [
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 
    'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 
    'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 
    'can', 'cannot', 'could', 'couldn', "couldn't", 'did', 'didn', "didn't", 
    'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 
    'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', 
    "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 
    'he', "he'd", "he'll", "he's", 'her', 'here', "here's", 'hers', 
    'herself', 'him', 'himself', 'his', 'how', "how's", 'i', "i'd", "i'll", 
    "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it's", 
    'its', 'itself', "let's", 'me', 'more', 'most', 'mustn', "mustn't", 'my', 
    'myself', 'no', 'nor', 'not', 'of', 'off', 'on', 'once', 'only', 'or', 
    'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 
    'same', 'shan', "shan't", 'she', "she'd", "she'll", "she's", 'should', 
    'shouldn', "shouldn't", 'so', 'some', 'such', 'than', 'that', "that's", 
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 
    "there's", 'these', 'they', "they'd", "they'll", "they're", "they've", 
    'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 
    'was', 'wasn', "wasn't", 'we', "we'd", "we'll", "we're", "we've", 'were', 
    'weren', "weren't", 'what', "what's", 'when', "when's", 'where',
    "where's", 'which', 'while', 'who', "who's", 'whom', 'why', "why's", 
    'with', 'won', "won't", 'would', 'wouldn', "wouldn't", 'you', "you'd", 
    "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves'
] 
stop_words_set = set(stop_words)

In [4]:
import time

def timer(func):
    start = time.time()
    result = func()
    end = time.time()

    print(f"Execution time: {end - start:.4f} seconds \n")
    return result

In [5]:
import re
from pyspark.sql import SparkSession

def rdd_run():
    # Start a spark session and create spark context for making rdd
    spark = SparkSession.builder \
        .appName("word_count") \
        .master("local[*]") \
        .config("spark.python.worker.faulthandler.enabled", "true") \
        .getOrCreate()

    sc = spark.sparkContext

    # Defind the rdd
    rdd = sc.textFile("./around_the_world_in_80_days.txt")

    # Create words
    words = rdd.flatMap(lambda line: re.sub(r"[^\w']+", " ", line.lower()).split()) \
                .filter(lambda word: word not in stop_words_set) \
                .map(lambda word: (word, 1)) \
                .reduceByKey(lambda  w1, w2: w1 + w2) \
                .sortBy(lambda word: word[1], ascending=False)

    return words

words = timer(rdd_run)

for word, count in words.collect():
    print(f"'{word}' is said {count} times")

Execution time: 18.1109 seconds 

'fogg' is said 646 times
'passepartout' is said 424 times
'mr' is said 391 times
's' is said 282 times
'phileas' is said 256 times
'fix' is said 255 times
'said' is said 194 times
'one' is said 172 times
'aouda' is said 136 times
'master' is said 129 times
'time' is said 126 times
'train' is said 119 times
'upon' is said 119 times
'will' is said 113 times
'now' is said 110 times
'two' is said 108 times
'sir' is said 103 times
'hundred' is said 98 times
'twenty' is said 97 times
'well' is said 96 times
'replied' is said 93 times
'steamer' is said 91 times
'hours' is said 90 times
'day' is said 89 times
'thousand' is said 88 times
'made' is said 87 times
'like' is said 83 times
'man' is said 80 times
'without' is said 80 times
'days' is said 78 times
'going' is said 76 times
'left' is said 76 times
'chapter' is said 74 times
'great' is said 73 times
'detective' is said 71 times
'london' is said 71 times
'passed' is said 71 times
'five' is said 71 times
'

# 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [8]:
# Set
stop_words_fr = [
    'au', 'aux', 'avec', 'ce', 'ces', 'dans', 'de', 'des', 'du', 'elle', 
    'en', 'et', 'eux', 'il', 'ils', 'je', 'la', 'le', 'les', 'leur', 
    'lui', 'ma', 'mais', 'me', 'même', 'mes', 'moi', 'mon', 'ne', 'nos', 
    'notre', 'nous', 'on', 'ou', 'par', 'pas', 'pour', 'qu', 'que', 'qui', 
    'sa', 'se', 'ses', 'son', 'sur', 'ta', 'te', 'tes', 'toi', 'ton', 
    'tu', 'un', 'une', 'vos', 'votre', 'vous', 'c', 'd', 'j', 'l', 'à',
    'm', 'n', 's', 't', 'y', 'été', 'étée', 'étés', 'étées', 'étant', 
    'étant', 'suis', 'es', 'est', 'sommes', 'êtes', 'sont', 'serai', 'seras', 'sera', 
    'serons', 'serez', 'seront', 'serais', 'serait', 'serions', 'seriez', 'seraient', 'étais', 'était', 
    'étions', 'étiez', 'étaient', 'fus', 'fut', 'fûmes', 'fûtes', 'furent', 'sois', 'soit', 
    'soyons', 'soyez', 'soient', 'fusse', 'fusses', 'fût', 'fussions', 'fussiez', 'fussent', 'ai', 
    'as', 'a', 'avons', 'avez', 'ont', 'aurai', 'auras', 'aura', 'aurons', 
    'aurez', 'auront', 'aurais', 'aurait', 'aurions', 'auriez', 'auraient', 'avais', 'avait', 
    'avions', 'aviez', 'avaient', 'eut', 'eûmes', 'eûtes', 'eurent', 'aie', 'aies', 
    'ait', 'ayons', 'ayez', 'aient', 'eusse', 'eusses', 'eût', 'eussions', 'eussiez', 
    'eussent'
]
stop_words_fr_set = set(stop_words_fr)

In [9]:
import re
from pyspark.sql import SparkSession

def rdd_run():
    # Start a spark session and create spark context for making rdd
    spark = SparkSession.builder \
        .appName("word_count") \
        .master("local[*]") \
        .config("spark.python.worker.faulthandler.enabled", "true") \
        .getOrCreate()

    sc = spark.sparkContext

    # Defind the rdd
    rdd_fr = sc.textFile("./le_tour_du_monde_en_80_jours.txt")

    # Create words
    words_fr = rdd_fr.flatMap(lambda line: re.sub(r"[^\w']+", " ", line.lower()).split()) \
                .filter(lambda word: word not in stop_words_fr_set) \
                .map(lambda word: (word, 1)) \
                .reduceByKey(lambda  w1, w2: w1 + w2) \
                .sortBy(lambda word: word[1], ascending=False)

    return words_fr

words_fr = timer(rdd_run)

for word, count in words_fr.collect():
    print(f"'{word}' is said {count} times")

Execution time: 9.1334 seconds 

'fogg' is said 689 times
'passepartout' is said 460 times
'plus' is said 341 times
'phileas' is said 332 times
'mr' is said 287 times
'fix' is said 286 times
'cette' is said 279 times
'heures' is said 243 times
'répondit' is said 215 times
'tout' is said 197 times
'bien' is said 196 times
'qu'il' is said 194 times
'si' is said 191 times
'dit' is said 183 times
'sans' is said 176 times
'comme' is said 173 times
'deux' is said 153 times
'monsieur' is said 145 times
'quelques' is said 136 times
'aouda' is said 134 times
'd'un' is said 131 times
'mrs' is said 131 times
'après' is said 124 times
'c'est' is said 123 times
'maître' is said 123 times
'c'était' is said 120 times
'là' is said 120 times
'train' is said 115 times
'dont' is said 115 times
'donc' is said 114 times
'ni' is said 110 times
'mille' is said 108 times
'temps' is said 107 times
'd'une' is said 106 times
'quand' is said 104 times
'vingt' is said 103 times
'encore' is said 103 times
'jours' i